> ⚠️ **পুরনো নোটবুক।** নতুন ও উন্নত ভার্সন ব্যবহার করুন: **`colab_bangla_english.ipynb`**
> — সেটায় ১–৫ ঘণ্টার টাইম-বাজেট, বাংলা+ইংরেজি অটো ডাটা-মিক্স, চেকপয়েন্ট/রিজিউম আর চ্যাট-টেস্ট সেল আছে।

# 🎓 MY-AI ট্রেনিং নোটবুক (Google Colab) — বিগিনার ভার্সন

এই নোটবুকে **একটা কমান্ডেই** সব অটো হবে: Hugging Face ডাটা ডাউনলোড → নিজের ডাটা মিশানো → কনভার্ট → LoRA ট্রেনিং → GGUF এক্সপোর্ট → টেস্ট।

**শুরু করার আগে (একবারই):**
1. `Runtime → Change runtime type → T4 GPU` সিলেক্ট করুন
2. সেলগুলো **উপর থেকে নিচে** চালান — কোনো সেল বাদ দেবেন না

**ফ্লো:**
```
তিনটা ফাইল আপলোড → GPU চেক → প্যাকেজ ইনস্টল
        ↓
একটা ট্রেনিং কমান্ড (ডাটা অটো প্রসেস হয়)
        ↓
zip ডাউনলোড → Ollama-তে চালান
```

In [ ]:
# ✅ GPU আছে কিনা দেখুন — শেষে "Tesla T4" দেখালে OK
!nvidia-smi

In [ ]:
# 📦 দরকারি প্যাকেজ ইনস্টল (২-৩ মিনিট লাগে, একবারই)
!pip install -q unsloth "unsloth[colab-new]" datasets transformers trl
print("✅ ইনস্টল শেষ")

In [ ]:
# 📤 এখন এই ২টা ফাইল আপলোড করুন (অবশ্যই):
#    - train_lora.py
#    - build_from_hf.py
#
# চাইলে সাথে এগুলোও (অপশনাল, বেশি ভালো ফল দেয়):
#    - myai-dataset.jsonl                      ← MY-AI ড্যাশবোর্ড থেকে Export করা নিজের চ্যাট
#    - bangla-english-banglish-chat.jsonl     ← রিপোর data/import/ ফোল্ডারের রেডিমেড বাংলা-ইংরেজি-বাংলিশ ডাটা
#
# ফাইলগুলো আগে নিজের কম্পিউটারে একটা ফোল্ডারে নামিয়ে রাখুন, তারপর নিচের সেল চালিয়ে
# সবগুলো একসাথে (Ctrl/Cmd চেপে) সিলেক্ট করে আপলোড করুন।
from google.colab import files
print("ফাইলগুলো বাছাই করুন...")
uploaded = files.upload()
print("\n✅ আপলোড শেষ:", list(uploaded.keys()))

---

## 🚀 এক কমান্ডে অটো ট্রেনিং

নিচের সেলটা চালালেই সব হবে — আলাদা করে ডাটা প্রসেস করার দরকার নেই:
1. Hugging Face থেকে UltraChat-এর ৫ হাজার কথোপকথন নামবে (ইংরেজি চ্যাট)
2. আপনার আপলোড করা নিজের ডাটা (বাংলা/বাংলিশ) অটো মিশে যাবে
3. ট্রেনিং শেষে GGUF বের হবে + একটা বাংলা প্রশ্ন দিয়ে মডেল টেস্ট হবে

**মডেল বাছাই (শুরুতেই এটা বদলাতে পারেন):**
| মডেল | নোট |
|---|---|
| `sarvamai/sarvam-1` | 🇧🇩 বাংলা+ইংরেজি — ডিফল্ট ✅ |
| `unsloth/Qwen2.5-3B-Instruct` | বাংলা ফলব্যাক, শক্তিশালী |
| `unsloth/Llama-3.2-1B-Instruct` | শুধু ইংরেজি, সবচেয়ে হালকা |

In [ ]:
# ⚡ সবকিছু একসাথে — ডাটা ডাউনলোড → মিশানো → ট্রেনিং → GGUF → টেস্ট
!python train_lora.py \
    --model sarvamai/sarvam-1 \
    --hf-dataset HuggingFaceH4/ultrachat_200k \
    --hf-split train_sft \
    --hf-limit 5000 \
    --hf-streaming \
    --extra-jsonl bangla-english-banglish-chat.jsonl \
    --extra-jsonl myai-dataset.jsonl \
    --epochs 2 \
    --output ./my-ai-model \
    --export-gguf \
    --test-prompt "আপনি কেমন আছেন?"

# নোট:
# - কোনো ফাইল আপলোড না করলে সেই --extra-jsonl লাইনটা মুছে দিন (ক্র্যাশ হবে না, skip করবে)
# - ডাটা বাড়াতে চাইলে --hf-limit 10000 করুন (সময় একটু বেশি লাগবে)
# - মডেল বদলাতে চাইলে --model unsloth/Qwen2.5-3B-Instruct দিন

In [ ]:
# ⬇️ ট্রেন করা মডেল ডাউনলোড করুন (zip আকারে)
!zip -r -q my-ai-model.zip my-ai-model
from google.colab import files
files.download('my-ai-model.zip')
print("✅ ডাউনলোড শুরু হয়েছে — GGUF ফাইলটা নিজের PC-তে রাখুন")
print("👉 এরপর: ollama create my-ai -f Modelfile && ollama run my-ai")
print("   (বিস্তারিত: training/BEGINNER_GUIDE_BN.md — ধাপ ৪ ও ৫)")

## 💡 সত্যি কথাটা

ফাইন-টিউন করলে মডেল আপনার **ঢং/ফরম্যাট** শিখবে, কিন্তু নতুন ফ্যাক্ট-জ্ঞান বাড়বে না। নতুন ফ্যাক্ট/ডকুমেন্ট শেখাতে চাইলে MY-AI-এর **Knowledge ট্যাব (RAG)** ব্যবহার করুন।

দুটো একসাথে ব্যবহার করলে সেরা: **LoRA = আপনার ঢং**, **RAG = আপনার তথ্য**।